In [0]:
spark.sql("""
SELECT *
FROM workspace.ecommerce.silver_events
WHERE event_type = 'purchase'
""").explain(True)


== Parsed Logical Plan ==
'Project [*]
+- 'Filter ('event_type = purchase)
   +- 'UnresolvedRelation [workspace, ecommerce, silver_events], [], false

== Analyzed Logical Plan ==
event_time: timestamp, event_type: string, product_id: int, category_id: bigint, category_code: string, brand: string, price: double, user_id: int, user_session: string, ingestion_ts: timestamp, event_date: date, price_tier: string
Project [event_time#13198, event_type#13199, product_id#13200, category_id#13201L, category_code#13202, brand#13203, price#13204, user_id#13205, user_session#13206, ingestion_ts#13207, event_date#13208, price_tier#13209]
+- Filter (event_type#13199 = purchase)
   +- SubqueryAlias workspace.ecommerce.silver_events
      +- Relation workspace.ecommerce.silver_events[event_time#13198,event_type#13199,product_id#13200,category_id#13201L,category_code#13202,brand#13203,price#13204,user_id#13205,user_session#13206,ingestion_ts#13207,event_date#13208,price_tier#13209] parquet

== Optimized

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.ecommerce.silver_events_part
USING DELTA
PARTITIONED BY (event_date, event_type)
AS
SELECT *
FROM workspace.ecommerce.silver_events
""")


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
import time

start = time.time()
spark.sql("""
SELECT *
FROM workspace.ecommerce.silver_events
WHERE event_type = 'purchase'
""").count()
print(f"Unpartitioned time: {time.time() - start:.2f}s")


Unpartitioned time: 1.91s


In [0]:
start = time.time()
spark.sql("""
SELECT *
FROM workspace.ecommerce.silver_events_part
WHERE event_type = 'purchase'
""").count()
print(f"Partitioned time: {time.time() - start:.2f}s")


Partitioned time: 0.82s


In [0]:
cached = spark.table("workspace.ecommerce.silver_events")
cached.count()   # Materialize count without caching

42341904

In [0]:
start = time.time()
cached.filter("event_type = 'purchase'").count()
print(f"Cached query time: {time.time() - start:.2f}s")


Cached query time: 0.76s
